# 03 - Token 消耗分析

**目的**: 深度分析各 Provider 的 Token 消耗模式，识别优化机会，辅助成本控制决策  
**数据来源**: 02-provider-comparison.ipynb 生成的 CSV 文件（或单独运行直接采集）  
**输出**: Token 分布图表 + 成本估算 + Prompt 压缩机会分析 + 优化建议

**使用方式**:
- 方式一：先运行 `02-provider-comparison.ipynb` 生成 CSV，再运行本 Notebook（推荐）
- 方式二：直接采集模式，配置下方参数后独立运行

## 0. 配置参数（按需修改）

In [ ]:
import os
from datetime import datetime
import glob

# =============================================
# 数据源配置
# =============================================

# 结果目录
DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)

# 方式一：自动读取最新的 02-provider-comparison 输出
# 若为 None，则进入直接采集模式
AUTO_LOAD_CSV = True

# 方式二：直接采集模式配置（仅在 AUTO_LOAD_CSV=False 时生效）
DIRECT_COLLECT = {
    "n_iterations": 5,   # 每个 Provider 采集轮数（建议 5-10）
    "test_prompt": "我需要为研发部门采购100台Dell PowerEdge R740服务器，配置双路Intel Xeon Gold 6248R处理器，512GB内存，用于AI模型训练，预算800万元，需要在2026年3月31日前交货，要求原厂5年保修。请帮我整理采购需求。",
    "max_tokens": 800,
}

# =============================================
# Provider 配置（直接采集模式使用）
# =============================================
APIPRO_BASE_URL = os.getenv("APIPRO_BASE_URL", "https://api.apipro.ai/v1")
APIPRO_API_KEY = os.getenv("APIPRO_API_KEY", "")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

PROVIDERS = [
    {
        "id": "apipro_claude",
        "label": "APIPro/Claude Sonnet",
        "type": "apipro",
        "model": "claude-sonnet-4-5-20250929",
        "cost_per_1k_input": 0.003,
        "cost_per_1k_output": 0.015,
        "enabled": bool(APIPRO_API_KEY),
    },
    {
        "id": "apipro_qwen",
        "label": "APIPro/Qwen Plus",
        "type": "apipro",
        "model": "qwen-plus",
        "cost_per_1k_input": 0.0004,
        "cost_per_1k_output": 0.0012,
        "enabled": bool(APIPRO_API_KEY),
    },
    {
        "id": "apipro_deepseek",
        "label": "APIPro/DeepSeek Chat",
        "type": "apipro",
        "model": "deepseek-chat",
        "cost_per_1k_input": 0.00014,
        "cost_per_1k_output": 0.00028,
        "enabled": bool(APIPRO_API_KEY),
    },
    {
        "id": "ollama_14b",
        "label": "Ollama/qwen2.5:14b",
        "type": "ollama",
        "model": "qwen2.5:14b",
        "cost_per_1k_input": 0.0,
        "cost_per_1k_output": 0.0,
        "enabled": True,
    },
    {
        "id": "ollama_7b",
        "label": "Ollama/qwen2.5:7b",
        "type": "ollama",
        "model": "qwen2.5:7b",
        "cost_per_1k_input": 0.0,
        "cost_per_1k_output": 0.0,
        "enabled": True,
    },
]

# 输出文件
RESULTS_FILE = f"{DATA_DIR}/token_analysis_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

print("配置加载完成")
print(f"数据目录: {DATA_DIR}")
print(f"AUTO_LOAD_CSV: {AUTO_LOAD_CSV}")
print(f"APIPro 已配置: {bool(APIPRO_API_KEY)}")

## 1. 依赖导入

In [ ]:
import httpx
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime

# 中文显示
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("依赖导入完成")

## 2. 数据加载（方式一：读取已有 CSV）

In [ ]:
df = None

if AUTO_LOAD_CSV:
    # 查找最新的 provider_comparison CSV
    csv_files = sorted(
        glob.glob(f"{DATA_DIR}/provider_comparison_*.csv"),
        reverse=True
    )

    if csv_files:
        latest_csv = csv_files[0]
        df = pd.read_csv(latest_csv)
        print(f"已加载: {latest_csv}")
        print(f"记录数: {len(df)}")
        print(f"列: {list(df.columns)}")
        df.head(3)
    else:
        print("未找到 provider_comparison CSV 文件")
        print("将切换到直接采集模式 (DIRECT_COLLECT)")
        AUTO_LOAD_CSV = False

if not AUTO_LOAD_CSV:
    print("直接采集模式 - 请继续执行下方采集单元格")

## 3. 直接采集模式（方式二：无 CSV 时）

若 `AUTO_LOAD_CSV=True` 且已加载 CSV，可跳过此节。

In [ ]:
def call_apipro_with_tokens(model: str, prompt: str, max_tokens: int, api_key: str, base_url: str) -> dict:
    """调用 APIPro API，返回延迟 + Token 用量"""
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": False,
    }
    start = time.perf_counter()
    try:
        resp = httpx.post(f"{base_url}/chat/completions", json=payload, headers=headers, timeout=60.0)
        elapsed = (time.perf_counter() - start) * 1000
        if resp.status_code == 200:
            data = resp.json()
            usage = data.get("usage", {})
            return {
                "success": True,
                "latency_ms": elapsed,
                "input_tokens": usage.get("prompt_tokens", 0),
                "output_tokens": usage.get("completion_tokens", 0),
                "total_tokens": usage.get("total_tokens", 0),
                "error": None,
            }
        return {"success": False, "latency_ms": elapsed, "input_tokens": 0, "output_tokens": 0, "total_tokens": 0, "error": resp.text[:200]}
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - start) * 1000, "input_tokens": 0, "output_tokens": 0, "total_tokens": 0, "error": str(e)}


def call_ollama_with_tokens(model: str, prompt: str, base_url: str) -> dict:
    """调用 Ollama API，返回延迟 + Token 用量"""
    # 先验证模型存在
    try:
        tags_resp = httpx.get(f"{base_url}/api/tags", timeout=5.0)
        if tags_resp.status_code == 200:
            models = [m["name"] for m in tags_resp.json().get("models", [])]
            if not any(model in m for m in models):
                return {"success": False, "latency_ms": 0, "input_tokens": 0, "output_tokens": 0, "total_tokens": 0, "error": f"Model {model} not found in Ollama"}
    except Exception as e:
        return {"success": False, "latency_ms": 0, "input_tokens": 0, "output_tokens": 0, "total_tokens": 0, "error": f"Ollama not reachable: {e}"}

    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
    }
    start = time.perf_counter()
    try:
        resp = httpx.post(f"{base_url}/api/chat", json=payload, timeout=120.0)
        elapsed = (time.perf_counter() - start) * 1000
        if resp.status_code == 200:
            data = resp.json()
            # Ollama token 统计在不同位置
            input_tokens = data.get("prompt_eval_count", 0)
            output_tokens = data.get("eval_count", 0)
            return {
                "success": True,
                "latency_ms": elapsed,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "total_tokens": input_tokens + output_tokens,
                "error": None,
            }
        return {"success": False, "latency_ms": elapsed, "input_tokens": 0, "output_tokens": 0, "total_tokens": 0, "error": resp.text[:200]}
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - start) * 1000, "input_tokens": 0, "output_tokens": 0, "total_tokens": 0, "error": str(e)}


def collect_token_data(providers: list, config: dict) -> pd.DataFrame:
    """直接采集模式：从各 Provider 收集 Token 数据"""
    results = []
    n = config["n_iterations"]
    prompt = config["test_prompt"]
    max_tokens = config["max_tokens"]

    print(f"开始直接采集: {n} 轮 × {len([p for p in providers if p['enabled']])} 个 Provider")
    print("=" * 60)

    for i in range(n):
        for prov in providers:
            if not prov["enabled"]:
                continue

            if prov["type"] == "apipro":
                r = call_apipro_with_tokens(prov["model"], prompt, max_tokens, APIPRO_API_KEY, APIPRO_BASE_URL)
            else:
                r = call_ollama_with_tokens(prov["model"], prompt, OLLAMA_BASE_URL)

            cost = (
                r["input_tokens"] / 1000 * prov["cost_per_1k_input"]
                + r["output_tokens"] / 1000 * prov["cost_per_1k_output"]
            ) if r["success"] else 0.0

            record = {
                "iteration": i + 1,
                "provider_id": prov["id"],
                "provider_label": prov["label"],
                "latency_ms": r["latency_ms"],
                "input_tokens": r["input_tokens"],
                "output_tokens": r["output_tokens"],
                "total_tokens": r["total_tokens"],
                "cost_usd": cost,
                "success": r["success"],
                "timestamp": datetime.now().isoformat(),
            }
            results.append(record)

            status = "OK" if r["success"] else "FAIL"
            print(f"[{status}] 轮{i+1:02d} | {prov['label']:30s} | "
                  f"in:{r['input_tokens']:5d} out:{r['output_tokens']:5d} | "
                  f"cost:${cost:.5f} | {r['latency_ms']:.0f}ms")

        if i < n - 1:
            time.sleep(2)

    return pd.DataFrame(results)


# 仅在非 AUTO_LOAD_CSV 模式下执行采集
if not AUTO_LOAD_CSV or df is None:
    df = collect_token_data(PROVIDERS, DIRECT_COLLECT)
    print(f"\n采集完成，共 {len(df)} 条记录")
else:
    print("已使用 CSV 数据，跳过直接采集")

## 4. Token 消耗统计

In [ ]:
# 只分析成功记录
df_ok = df[df["success"] == True].copy()

print("=" * 70)
print("Token 消耗统计 (成功请求)")
print("=" * 70)

# 按 Provider 分组统计
token_stats = df_ok.groupby("provider_label").agg(
    input_mean=("input_tokens", "mean"),
    input_std=("input_tokens", "std"),
    output_mean=("output_tokens", "mean"),
    output_std=("output_tokens", "std"),
    total_mean=("total_tokens", "mean"),
    cost_mean=("cost_usd", "mean"),
    cost_total=("cost_usd", "sum"),
    count=("success", "count")
).round(2)

print(token_stats.to_string())

print("\n" + "=" * 70)
print("输入/输出 Token 比值分析")
print("=" * 70)

for label, row in token_stats.iterrows():
    if row["output_mean"] > 0:
        ratio = row["input_mean"] / row["output_mean"]
        print(f"  {label:30s}: 输入/输出比 = {ratio:.2f}:1")
        print(f"  {'':30s}  输入 {row['input_mean']:.0f} tokens | 输出 {row['output_mean']:.0f} tokens")

## 5. 成本估算（按使用量规模）

In [ ]:
# 成本费率映射
cost_rates = {p["label"]: {"input": p["cost_per_1k_input"], "output": p["cost_per_1k_output"]} for p in PROVIDERS}

# 使用量规模（月请求数）
MONTHLY_VOLUMES = [
    ("小规模", 1000),
    ("中规模", 10000),
    ("大规模", 100000),
]

print("=" * 80)
print("月度成本估算 (USD)")
print("=" * 80)
print(f"{'Provider':30s}", end="")
for scale_label, _ in MONTHLY_VOLUMES:
    print(f"  {scale_label:>12s}", end="")
print()
print("-" * 80)

cost_table = {}
for label, row in token_stats.iterrows():
    costs = []
    rates = cost_rates.get(label, {"input": 0, "output": 0})
    for scale_label, volume in MONTHLY_VOLUMES:
        monthly_cost = volume * (
            row["input_mean"] / 1000 * rates["input"]
            + row["output_mean"] / 1000 * rates["output"]
        )
        costs.append(monthly_cost)
    cost_table[label] = costs
    print(f"{label:30s}", end="")
    for c in costs:
        if c == 0:
            print(f"  {'$0 (本地)':>12s}", end="")
        else:
            print(f"  ${c:>10.2f}", end="")
    print()

print("=" * 80)
print("注意: Ollama 为本地推理，电费和硬件折旧未计入")

## 6. 可视化

In [ ]:
providers_list = list(token_stats.index)
n_providers = len(providers_list)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f"Token 消耗与成本分析 (n={len(df_ok)})", fontsize=14)

# 图1: 输入/输出 Token 堆叠柱状图
ax1 = axes[0, 0]
x = range(n_providers)
bars_in = ax1.bar(x, token_stats["input_mean"], label="输入 Token", color="steelblue", alpha=0.8)
bars_out = ax1.bar(x, token_stats["output_mean"], bottom=token_stats["input_mean"], label="输出 Token", color="coral", alpha=0.8)
ax1.set_xticks(list(x))
ax1.set_xticklabels(providers_list, rotation=20, ha="right", fontsize=8)
ax1.set_ylabel("Token 数")
ax1.set_title("输入/输出 Token 分布")
ax1.legend()

# 在柱上标注数值
for i, (bar_in, bar_out) in enumerate(zip(bars_in, bars_out)):
    ax1.text(bar_in.get_x() + bar_in.get_width()/2, bar_in.get_height()/2,
             f"{int(token_stats['input_mean'].iloc[i])}", ha='center', va='center', fontsize=7, color='white')
    ax1.text(bar_out.get_x() + bar_out.get_width()/2,
             bar_out.get_y() + bar_out.get_height()/2,
             f"{int(token_stats['output_mean'].iloc[i])}", ha='center', va='center', fontsize=7, color='white')

# 图2: 每次请求成本（可见的 Provider）
ax2 = axes[0, 1]
cost_values = token_stats["cost_mean"].values
colors = ["steelblue" if c > 0 else "lightgray" for c in cost_values]
bars = ax2.bar(range(n_providers), cost_values, color=colors, alpha=0.8)
ax2.set_xticks(range(n_providers))
ax2.set_xticklabels(providers_list, rotation=20, ha="right", fontsize=8)
ax2.set_ylabel("成本 (USD/次)")
ax2.set_title("每次请求平均成本")
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.5f'))
for bar, val in zip(bars, cost_values):
    label = "$0\n(本地)" if val == 0 else f"${val:.5f}"
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(cost_values)*0.01,
             label, ha='center', va='bottom', fontsize=7)

# 图3: 月度成本对比（中规模 10000 次）
ax3 = axes[1, 0]
scale_idx = 1  # 中规模
monthly_costs = [cost_table[label][scale_idx] for label in providers_list]
colors3 = ["steelblue" if c > 0 else "lightgray" for c in monthly_costs]
ax3.bar(range(n_providers), monthly_costs, color=colors3, alpha=0.8)
ax3.set_xticks(range(n_providers))
ax3.set_xticklabels(providers_list, rotation=20, ha="right", fontsize=8)
ax3.set_ylabel("月度成本 (USD)")
ax3.set_title(f"月度成本估算 ({MONTHLY_VOLUMES[scale_idx][0]}: {MONTHLY_VOLUMES[scale_idx][1]:,} 次/月)")
ax3.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.2f'))

# 图4: 输出 Token 分布箱线图
ax4 = axes[1, 1]
data_for_box = [df_ok[df_ok["provider_label"] == lbl]["output_tokens"].values for lbl in providers_list]
data_for_box = [d for d in data_for_box if len(d) > 0]
labels_for_box = [lbl for lbl, d in zip(providers_list, [df_ok[df_ok["provider_label"] == l]["output_tokens"].values for l in providers_list]) if len(d) > 0]
ax4.boxplot(data_for_box, labels=[l.split("/")[-1] for l in labels_for_box], patch_artist=True)
ax4.set_ylabel("输出 Token 数")
ax4.set_title("输出 Token 分布（箱线图）")
ax4.tick_params(axis='x', rotation=15)

plt.tight_layout()
chart_file = f"{DATA_DIR}/token_analysis_chart_{datetime.now().strftime('%Y%m%d_%H%M')}.png"
plt.savefig(chart_file, dpi=150, bbox_inches="tight")
plt.show()
print(f"图表已保存: {chart_file}")

## 7. Prompt 压缩机会分析

In [ ]:
print("=" * 70)
print("Prompt 压缩机会分析")
print("=" * 70)

# 基准：使用输入 Token 最多的 Provider 作为分析基准
avg_input = token_stats["input_mean"].mean()
avg_output = token_stats["output_mean"].mean()

print(f"\n平均输入 Token: {avg_input:.0f}")
print(f"平均输出 Token: {avg_output:.0f}")
print(f"平均总 Token:   {avg_input + avg_output:.0f}")

print("\n--- 压缩机会 ---")

# 压缩场景分析
scenarios = [
    {
        "name": "System Prompt 精简（-20%）",
        "input_reduction": 0.20,
        "difficulty": "低",
        "risk": "低",
        "description": "移除冗余说明，使用简洁指令替代长描述"
    },
    {
        "name": "Few-shot 示例压缩（-30%）",
        "input_reduction": 0.30,
        "difficulty": "中",
        "risk": "中",
        "description": "减少示例数量，保留最具代表性的 1-2 个"
    },
    {
        "name": "历史对话截断（-40%）",
        "input_reduction": 0.40,
        "difficulty": "高",
        "risk": "高",
        "description": "仅保留最近 N 轮对话，配合摘要机制"
    },
]

for s in scenarios:
    saved_tokens = avg_input * s["input_reduction"]
    new_input = avg_input - saved_tokens
    print(f"\n{s['name']}")
    print(f"  描述: {s['description']}")
    print(f"  难度: {s['difficulty']} | 风险: {s['risk']}")
    print(f"  节省输入 Token: {saved_tokens:.0f} ({s['input_reduction']*100:.0f}%)")

    # 按 Provider 计算节省成本
    for label, row in token_stats.iterrows():
        rates = cost_rates.get(label, {"input": 0, "output": 0})
        saved_cost_per_req = saved_tokens / 1000 * rates["input"]
        if saved_cost_per_req > 0:
            saved_monthly_10k = saved_cost_per_req * 10000
            print(f"  [{label}] 节省 ${saved_cost_per_req:.5f}/次 = ${saved_monthly_10k:.2f}/月(1万次)")

print("\n" + "=" * 70)
print("结论")
print("=" * 70)
print("1. 对于 Ollama（本地）：Token 压缩主要影响推理速度，与成本无关")
print("2. 对于 APIPro 付费 Provider：System Prompt 精简是性价比最高的优化")
print("3. 输出 Token 由模型决定，压缩空间有限；重点优化输入 Token")

## 8. 综合优化建议

In [ ]:
print("=" * 70)
print("综合优化建议")
print("=" * 70)

# 计算关键指标
if len(token_stats) > 0:
    cheapest = token_stats["cost_mean"].idxmin() if token_stats["cost_mean"].max() > 0 else None
    most_tokens = token_stats["output_mean"].idxmax()
    most_efficient = token_stats["output_mean"].idxmax()  # 输出多通常质量更高

    print("\n[成本维度]")
    if cheapest:
        print(f"  最低成本 Provider: {cheapest}")
        print(f"  建议: 高频次、低价值任务优先使用")
    free_providers = [lbl for lbl in providers_list if token_stats.loc[lbl, "cost_mean"] == 0]
    if free_providers:
        print(f"  零成本选项: {', '.join(free_providers)} (需本地 GPU/CPU)")

    print("\n[质量维度]")
    print(f"  输出最丰富 Provider: {most_tokens} (输出 {token_stats.loc[most_tokens, 'output_mean']:.0f} tokens 均值)")
    print(f"  建议: 文档生成、复杂分析任务优先使用")

    print("\n[架构建议]")
    print("  1. 双层路由策略:")
    print("     - 简单意图识别 → Ollama 本地（零成本）")
    print("     - 复杂文档生成 → APIPro Claude（高质量）")
    print("  2. Token 缓存:")
    print("     - System Prompt 不变部分启用 Prompt Caching（Claude 支持）")
    print("     - 预期节省 60-80% System Prompt Token 费用")
    print("  3. 异步批处理:")
    print("     - 非实时任务（文档生成）使用批量 API")
    print("     - 预期节省 50% 成本（Claude Batch API 半价）")
    print("  4. 降级策略:")
    print("     - APIPro 限流时自动降级至 Ollama")
    print("     - 避免请求失败影响用户体验")

print("\n" + "=" * 70)

## 9. 保存结果

In [ ]:
# 保存详细数据
df.to_csv(RESULTS_FILE, index=False, encoding="utf-8")
print(f"数据已保存: {RESULTS_FILE}")

# 保存统计摘要
summary_file = RESULTS_FILE.replace(".csv", "_summary.csv")
token_stats.to_csv(summary_file, encoding="utf-8")
print(f"统计摘要已保存: {summary_file}")

print(f"\n共 {len(df)} 条记录，{len(df_ok)} 条成功")
df_ok.groupby("provider_label")[["input_tokens", "output_tokens", "cost_usd"]].mean().round(4)